# 1、ConversationTokenBufferMemory的使用

示例1：

In [3]:
#1.导入相关包
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")
# 3.定义ConversationTokenBufferMemory对象
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=10  # 设置token上限 默认值为2000
)
# 添加对话
memory.save_context({"input": "你好吗？"}, {"output": "我很好，谢谢！"})
memory.save_context({"input": "今天天气如何？"}, {"output": "晴天，25度"})
# 查看当前记忆
print(memory.load_memory_variables({}))

{'history': ''}


In [5]:
#1.导入相关包
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")
# 3.定义ConversationTokenBufferMemory对象
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=20   # 设置token上限 默认值为2000
)
# 添加对话
memory.save_context({"input": "你好吗？"}, {"output": "我很好，谢谢！"})
memory.save_context({"input": "今天天气如何？"}, {"output": "晴天，25度"})
# 查看当前记忆
print(memory.load_memory_variables({}))

{'history': 'AI: 晴天，25度'}


# 2、ConversationSummaryMemory的使用

示例1：

如果实例化ConversationSummaryMemory前，没有历史消息，可以使用构造方法实例化

In [7]:
# 1.导入相关包
from langchain_classic.memory import ConversationSummaryMemory, ChatMessageHistory
from langchain_openai import ChatOpenAI
# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")
# 3.定义ConversationSummaryMemory对象
memory = ConversationSummaryMemory(llm=llm)
# 4.存储消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手小智"})
memory.save_context({"input": "初次对话，你能介绍一下你自己吗？"}, {"output": "当然可以了。我是一个无所不能的小智。"})
# 5.读取消息（总结后的）
print(memory.load_memory_variables({}))

{'history': 'The human greets the AI with "hello," and the AI responds by asking, "What\'s wrong?" The human inquires, "Who are you?" to which the AI replies, "I am AI assistant Xiaozhi." The human then asks for more information, and the AI introduces itself as an all-powerful assistant named Xiaozhi.'}


示例2：如果实例化ConversationSummaryMemory前，已经有历史消息，可以调用from_messages()实例化

In [9]:
from langchain_classic.memory import ConversationSummaryMemory, ChatMessageHistory
from langchain_openai import ChatOpenAI
# 2.定义ChatMessageHistory对象
llm = ChatOpenAI(model="gpt-4o-mini")
# 3.假设原始消息
history = ChatMessageHistory()
history.add_user_message("你好，你是谁？")
history.add_ai_message("我是AI助手小智")

# 4、创建ConversationSummaryMemory实例
memory = ConversationSummaryMemory.from_messages(
    llm = llm,
    chat_memory = history,
)
print(memory.load_memory_variables({}))
memory.save_context(inputs = {"human":"我的名字是小明"},outputs = {"AI":{"很高兴认识你"}})
print(memory.load_memory_variables({}))

{'history': 'The human greets the AI and asks who it is. The AI responds that it is the AI assistant named Xiao Zhi.'}
{'history': 'The human greets the AI and asks who it is. The AI responds that it is the AI assistant named Xiao Zhi. The human introduces themselves as Xiao Ming, and the AI expresses that it is pleased to meet them.'}


# 3、 ConversationSummaryBufferMemory的使用

在保留最近对话原始记录的同时，对较早的对话内容进行智能摘要（完整对话记录+摘要记忆）

示例1：

In [11]:
from langchain_classic.memory import ConversationSummaryBufferMemory

llm =ChatOpenAI(model = "gpt-4o-mini")
memory = ConversationSummaryBufferMemory(
    llm = llm,
    max_token_limit=40, #控制缓冲区大小
    return_messages=True
)
# 向memory存储信息
memory.save_context(inputs = {"input":"你好，我的名字叫小明"},outputs = {"output":"很高兴认识你"})
memory.save_context(inputs = {"input":"李白是哪个朝代的"},outputs = {"output":"李白是唐朝的"})
memory.save_context(inputs = {"input":"唐宋八大家有苏轼吗"},outputs = {"output":"有"})

print(memory.load_memory_variables({}))

{'history': [SystemMessage(content='The human introduces themselves as 小明. The AI expresses pleasure in meeting them, and the human asks which dynasty Li Bai belonged to.', additional_kwargs={}, response_metadata={}), AIMessage(content='李白是唐朝的', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='唐宋八大家有苏轼吗', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


示例2：模拟客服交互

In [13]:
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.llm import LLMChain
# 1、初始化大语言模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=500 # 大模型输出的最大token数目
)
# 2、定义提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服助手，用中文友好回复用户问题。保持专业但亲切的语气。"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])
# 3、创建带摘要缓冲的记忆系统
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=400,
    memory_key="chat_history",
    return_messages=True
)
# 4、创建对话链
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
)
# 5、模拟多轮对话
dialogue = [
    ("你好，我想查询订单12345的状态", None),
    ("这个订单是上周五下的", None),
    ("我现在急着用，能加急处理吗", None),
    ("等等，我可能记错订单号了，应该是12346", None),
    ("对了，你们退货政策是怎样的", None)
]

# 6、执行对话
for user_input, _ in dialogue:
    response = chain.invoke({"input": user_input})
    print(f"用户: {user_input}")
    print(f"客服: {response['text']}\n")
# 7、查看当前记忆状态
print("\n=== 当前记忆内容 ===")
print(memory.load_memory_variables({}))

用户: 你好，我想查询订单12345的状态
客服: 你好！感谢你的咨询。关于订单12345的状态，我会为你查询一下。请稍等片刻。 

（如果有具体的查询结果，可以在这里提供给用户。如果没有，可以回复：） 

目前我无法直接查看订单状态，但你可以登录我们的官网，在“我的订单”中查看详细信息。如果有其他问题，欢迎随时问我！

用户: 这个订单是上周五下的
客服: 谢谢你的补充信息！根据你提供的订单时间，上周五下的订单通常会在1-3个工作日内处理和发货。请你稍等，我会尽快为你确认具体的订单状态。

如果你有其他问题或需要进一步的帮助，请随时告诉我！

用户: 我现在急着用，能加急处理吗
客服: 我理解你的着急心情！关于加急处理的请求，通常情况下我们会尽量满足客户的需求，但具体操作还需要根据订单的处理流程和物流情况来决定。

请你提供一下订单的具体需求，我会尽快向相关部门反馈，看看是否可以加急处理。同时，建议你也可以查看一下订单状态，看看是否已经发货。

谢谢你的理解，如果有其他问题，请随时告诉我！

用户: 等等，我可能记错订单号了，应该是12346
客服: 没问题！感谢你更新订单号。让我为你查询订单12346的状态，请稍等片刻。

（如果有具体的查询结果，可以在这里提供给用户。如果没有，可以回复：）

我会尽快为你确认订单状态。如果有其他问题或需要进一步的帮助，请随时告诉我！

用户: 对了，你们退货政策是怎样的
客服: 我们的退货政策如下：

1. **退货时间**：一般情况下，自收到商品之日起，您可以在7天内申请退货。
   
2. **退货条件**：商品必须保持未使用状态，且包装完好，附带完整的配件和赠品。对于部分特殊商品（如内衣、食品等），可能不支持退货。

3. **申请流程**：
   - 登录您的账户，找到“我的订单”。
   - 选择需要退货的订单，点击“申请退货”。
   - 按照系统提示填写相关信息，并提交申请。

4. **退款方式**：退款会按照您原支付方式返还，处理时间一般为7-14个工作日。

如果您有具体的商品需要退货或还有其他问题，请随时告诉我，我会尽力帮您解决！


=== 当前记忆内容 ===
{'chat_history': [SystemMessage(content='The human inquires about the s

# 4、ConversationEntityMemory(了解)

In [15]:
from langchain_classic.chains.conversation.base import LLMChain
from langchain_classic.memory import ConversationEntityMemory
from langchain_classic.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE
from langchain_openai import ChatOpenAI
# 初始化大语言模型
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0
)
# 使用LangChain为实体记忆设计的预定义模板
prompt = ENTITY_MEMORY_CONVERSATION_TEMPLATE
# 初始化实体记忆
memory = ConversationEntityMemory(llm=llm)
# 提供对话链
chain = LLMChain(
    llm=llm,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
    memory=memory,
    #verbose=True,  # 设置为True可以看到链的详细推理过程
)
# 进行几轮对话，记忆组件会在后台自动提取和存储实体信息
chain.invoke(input="你好，我叫蜘蛛侠。我的好朋友包括钢铁侠、美国队长和绿巨人。")
chain.invoke(input="我住在纽约。")
chain.invoke(input="我使用的装备是由斯塔克工业提供的。")
# 查询记忆体中存储的实体信息
print("\n当前存储的实体信息:")
print(chain.memory.entity_store.store)
# 基于记忆进行提问
answer = chain.invoke(input="你能告诉我蜘蛛侠住在哪里以及他的好朋友有哪些吗？")
print("\nAI的回答:")
print(answer)


当前存储的实体信息:
{'蜘蛛侠': '蜘蛛侠的好朋友包括钢铁侠、美国队长和绿巨人。', '钢铁侠': '钢铁侠是蜘蛛侠的好朋友之一。', '美国队长': '美国队长是蜘蛛侠的好朋友之一。', '绿巨人': '绿巨人是蜘蛛侠的好朋友之一。', '纽约': '蜘蛛侠住在纽约。', '斯塔克工业': '斯塔克工业提供蜘蛛侠使用的装备。'}

AI的回答:
{'input': '你能告诉我蜘蛛侠住在哪里以及他的好朋友有哪些吗？', 'history': 'Human: 你好，我叫蜘蛛侠。我的好朋友包括钢铁侠、美国队长和绿巨人。\nAI: 你好，蜘蛛侠！很高兴认识你。你和钢铁侠、美国队长以及绿巨人都是超级英雄，真是一个强大的团队！你们最近有什么冒险吗？\nHuman: 我住在纽约。\nAI: 纽约是一个充满活力的城市，适合超级英雄们活动！你在纽约的生活怎么样？有没有遇到什么有趣的事情或者挑战？\nHuman: 我使用的装备是由斯塔克工业提供的。\nAI: 那真是太棒了！斯塔克工业的科技非常先进，钢铁侠的装备也非常酷。你最喜欢斯塔克工业的哪一项装备？或者有没有什么特别的功能让你觉得特别有用？', 'entities': {'蜘蛛侠': '蜘蛛侠的好朋友包括钢铁侠、美国队长和绿巨人。'}, 'text': '蜘蛛侠住在纽约市，这是一个充满活力和挑战的地方。他的好朋友包括钢铁侠、美国队长和绿巨人，他们都是超级英雄，常常一起合作对抗各种威胁。蜘蛛侠与他们之间的友谊非常深厚，彼此之间也有很多精彩的冒险故事！你对他们的关系有什么特别想了解的吗？'}


# 5、ConversationKGMemory(了解)

知识图谱：不仅能识别和存储实体，还能捕捉实体之间的复杂关系，形成结构化的知识网络。

In [17]:
#1.导入相关包
from langchain_classic.memory import ConversationKGMemory
from langchain_openai import ChatOpenAI
# 2.定义LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
# 3.定义ConversationKGMemory对象
memory = ConversationKGMemory(llm=llm)
# 4.保存会话
memory.save_context(inputs = {"input": "向山姆问好"}, outputs = {"output": "山姆是谁"})
memory.save_context({"input": "山姆是我的朋友"}, {"output": "好的"})
# 5.查询会话
memory.load_memory_variables({"input": "山姆是谁"})

{'history': 'On 山姆: 山姆 是 我的朋友.'}

In [18]:
memory.get_knowledge_triplets("她最喜欢的颜色是红色") #将对话内容转化为 (头实体, 关系, 尾实体) 的三元组形式

[KnowledgeTriple(subject='山姆', predicate='是', object_='我的朋友'),
 KnowledgeTriple(subject='山姆', predicate='最喜欢的颜色', object_='红色')]

# 6、VectorStoreRetrieverMemory(了解)

适用场景：这种记忆特别适合需要长期记忆和语义理解的复杂对话系统。

In [21]:
# 1.导入相关包
from langchain_openai import OpenAIEmbeddings
from langchain_classic.memory import VectorStoreRetrieverMemory
from langchain_community.vectorstores import FAISS
from langchain_classic.memory import ConversationBufferMemory
# 2.定义ConversationBufferMemory对象
memory = ConversationBufferMemory()
memory.save_context({"input": "我最喜欢的食物是披萨"}, {"output": "很高兴知道"})
memory.save_context({"Human": "我喜欢的运动是跑步"}, {"AI": "好的,我知道了"})
memory.save_context({"Human": "我最喜欢的运动是足球"}, {"AI": "好的,我知道了"})
# 3.定义向量嵌入模型
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-ada-002" )
# 4.初始化向量数据库
vectorstore = FAISS.from_texts(memory.buffer.split("\n"), embeddings_model)# 空初始化
# 5.定义检索对象
retriever = vectorstore.as_retriever(search_kwargs=dict(k=1))
# 6.初始化VectorStoreRetrieverMemory
memory = VectorStoreRetrieverMemory(retriever=retriever)
print(memory.load_memory_variables({"prompt":"我最喜欢的食物是"}))

{'history': 'Human: 我最喜欢的食物是披萨'}
